# IWI experiment · Ridge regression on the NL features of 06 and the Landsat features of 07

For every test fold and every row of 09, this notebook joins the 512 NL features of the 08 model and the 512 Landsat features of the 09 model of that fold (the layers before their IWI outputs), fits a ridge regression of IWI on the clusters of the training and validation folds, and reports the R², RMSE and MAE on the test fold. As in `29_RidgeRegression_Landsat-NL`, the features of test fold f come from the networks trained without fold f (model `test_f`), so neither the networks nor the ridge have seen the test clusters.

* **Inputs:** `test_<fold>/features.csv` of every model of 08 (`nl_feat_0` … `nl_feat_511`) and 09 (`ls_feat_0` … `ls_feat_511`), joined on `cluster_id`. The clusters in both files are used (the clusters of 09, which also have a valid NL image); `role` (train, val, test) comes from the files and must agree. Clusters without IWI are left out.
* **Ridge:** the features are standardised with the mean and standard deviation of the clusters the ridge is fitted on (`STANDARDIZE`), and the intercept is not penalised. The penalty `alpha` is chosen on the validation fold: a ridge is fitted on the training folds for every alpha of `ALPHAS`, and the alpha with the lowest validation MSE is kept. Then the ridge with that alpha is fitted on the training and validation folds together and tested on the test fold. `ALPHA = 1.0` with `STANDARDIZE = False` gives the setup of 29_RidgeRegression_Landsat-NL (raw features, alpha 1).
* **For comparison,** on the same test clusters: the same ridge on the NL features alone and on the Landsat features alone, and the IWI predicted by the two networks themselves.
* **Folds,** as in 08 and 09:

| test fold | ridge fitted on (training + validation folds) | Keras name |
|---|---|---|
| E | A, B, C + D | Fold12345 |
| A | B, C, D + E | Fold23451 |
| B | C, D, E + A | Fold34512 |
| C | D, E, A + B | Fold45123 |
| D | E, A, B + C | Fold51234 |

**Output,** in `ridge_runs/<RUN_NAME>/`: `ridge_per_fold.csv` (every row, test fold and feature set, with the chosen alpha), `ridge_results.csv` (mean and standard deviation over the test folds, and the R², RMSE and MAE of the five test folds pooled, also per period and for rural and urban clusters), `<key>_test_predictions.csv` (every test cluster with the predictions of all feature sets) `ridge_r2.png`, and the results per country, `ridge_per_country.csv` and `ridge_per_country.png` (section 5), and the NDVI screen of section 6, `ndvi_screen.csv`, `ndvi_screen_per_country.csv`, `ndvi_screen_correlations.csv`, `ndvi_screen_predictions.csv` and `ndvi_screen.png` (with the NDVI statistics of the band caches in `ridge_runs/ndvi_patch_stats_<cache>.csv`).

**Running.** The rows and folds whose features files exist in both `NL_RUNS` and `LANDSAT_RUNS` are used, so the notebook can run while 08 and 09 are still going and again when they are done. It needs no GPU. For the ViT features of 10, set `LANDSAT_RUNS`, `LANDSAT_PREFIX` and `RUN_NAME` in Settings.

> **Numbering.** This notebook was written before the notebooks were renumbered for this repository. Apart from the title, its text, comments and plot labels use the earlier numbers: 05 = `04_Table2_DHS`, 07 = `05_Table2_ResNet18`, 08 = `06_NL_ResNet18_Features`, 09 = `07_Landsat_ResNet18_Features`, 10 = `08_Landsat_ViTS16_Features`, 11 = `09_Ridge_NL_Landsat` and 12 = `10_Ridge_NL_ViT`; 01 to 03 are unchanged. The earlier 04 and 06 belonged to the simulated dataset of the preliminary study and are not included. Folder names such as `ridge_runs/nl08_landsat09` keep the earlier numbers.

# Import libraries

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator
from IPython.display import display

# Settings

In [ ]:
# ---- input / output ------------------------------------------------------------------------
WORK_DIR       = '/path/to/data/'
NL_RUNS        = os.path.join(WORK_DIR, 'nl_resnet18_runs', 'DMSP_VIIRS_scratch')       # 08: test_<fold>/features.csv
LANDSAT_RUNS   = os.path.join(WORK_DIR, 'landsat_resnet18_runs', 'DMSP_VIIRS_scratch')  # 09: <key>/test_<fold>/...
NL_PREFIX      = 'nl_feat_'
LANDSAT_PREFIX = 'ls_feat_'        # 10 (ViT): LANDSAT_RUNS = .../landsat_vits16_runs/DMSP_VIIRS_finetune, 'vit_feat_'
RUN_NAME       = 'nl08_landsat09'  # output folder ridge_runs/RUN_NAME; change it with LANDSAT_RUNS (10: 'nl08_vit10')
OUT_DIR        = os.path.join(WORK_DIR, 'ridge_runs', RUN_NAME)

# ---- ridge ----------------------------------------------------------------------------------------------------
TARGET      = 'iwi'
ALPHA       = 'val'                   # 'val': the alpha of ALPHAS with the lowest MSE on the validation fold (ridge
                                      # fitted on the training folds); a number: that alpha for every fold
                                      # (29_RidgeRegression_Landsat-NL: 1.0 with STANDARDIZE = False)
ALPHAS      = np.logspace(-2, 6, 33)  # 0.01 ... 1e6, four per decade
STANDARDIZE = True                    # standardise every feature with the clusters the ridge is fitted on

FOLDS = ['A', 'B', 'C', 'D', 'E']
KERAS_NAME = {'E': 'Fold12345', 'A': 'Fold23451', 'B': 'Fold34512', 'C': 'Fold45123', 'D': 'Fold51234'}  # 27_Train_NL
ROWS = {                              # the rows of 09: key -> input
    'landsat':          'Landsat images',
    'landsat_ndvi':     'Landsat images and temporal MODIS NDVI',
    'landsat_wavelet6': 'Landsat images and 6 statistical wavelet-based features',
    'landsat_hfc_max':  'Landsat images and max values of HFC',
    'landsat_hfc_mean': 'Landsat images and mean values of HFC',
    'landsat_hfc_min':  'Landsat images and min values of HFC',
    'landsat_lfc_max':  'Landsat images and max values of LFC',
    'landsat_lfc_mean': 'Landsat images and mean values of LFC',
    'landsat_lfc_min':  'Landsat images and min values of LFC',
}
FEATURE_SETS = ['NL + Landsat', 'NL', 'Landsat']                  # ridge on these features
NETWORKS = ['NL network (08)', 'Landsat network (09)']            # the IWI predicted by the networks themselves

# ---- results per country (section 5) ------------------------------------------------------------------------------
COUNTRY_ROW          = 'best'   # row of the per-country tables and plot: 'best' (the row of the plot in section 4)
                                # or a key of ROWS
MIN_COUNTRY_CLUSTERS = 30       # countries with fewer test clusters are left out of the tables and the plot (they
                                # stay in ridge_per_country.csv)
MIN_FOLD_CLUSTERS    = 10       # a country's R2 within one test fold needs at least this many of its clusters there

# ---- NDVI screen (section 6) --------------------------------------------------------------------------------------
NDVI_CACHES = {s: os.path.join(WORK_DIR, 'table1_cache', f'{s}_monthly_dmey_symmetric') for s in ('DMSP', 'VIIRS')}
FOLDS_CSV   = os.path.join(WORK_DIR, 'train_clusters_folds.csv')   # 03: its spatial groups, for the bootstrap
SCREEN_ROW  = 'landsat'                    # the row of the plot (its network has no NDVI); the tables show all rows
DARK_NL     = {'DMSP': 0.0, 'VIIRS': 0.3}  # dark cluster: the mean of its NL image (as downloaded) is at most this
BOOTSTRAP   = 1000                         # resamples of 03's spatial groups for the 95 % intervals (0: none)
N_WORKERS   = len(os.sched_getaffinity(0))  # processes that read the band caches (once, for the NDVI statistics)

print('NL features:', NL_RUNS, '\nLandsat features:', LANDSAT_RUNS, '\noutput:', OUT_DIR)
print('alpha:', 'chosen on the validation fold' if ALPHA == 'val' else ALPHA, '| standardised features:', STANDARDIZE,
      '| numpy BLAS threads:', os.environ.get('OMP_NUM_THREADS', 'default'))

# 1. The features files

A row and test fold are used when the features file of the 08 model and of the 09 model of that fold both exist.

In [ ]:
nl_file = lambda f: os.path.join(NL_RUNS, f'test_{f}', 'features.csv')
ls_file = lambda key, f: os.path.join(LANDSAT_RUNS, key, f'test_{f}', 'features.csv')
found = pd.DataFrame([[os.path.exists(nl_file(f)) for f in FOLDS]] +
                     [[os.path.exists(ls_file(k, f)) for f in FOLDS] for k in ROWS],
                     index=['NL (08)'] + list(ROWS), columns=[f'test {f}' for f in FOLDS])
TODO = [(k, f) for k in ROWS for f in FOLDS if os.path.exists(nl_file(f)) and os.path.exists(ls_file(k, f))]
print(f'{len(TODO)} of {len(ROWS) * len(FOLDS)} row and test fold combinations have both features files')
found.replace({True: 'yes', False: '-'})

# 2. Ridge regression

`ridge_predictions()` fits a ridge regression with intercept for many alphas at once, from one eigendecomposition of XᵀX (the features centred on the fitted clusters), and returns the predictions for other clusters. The cell checks it against scikit-learn's `Ridge`.

In [ ]:
def r2_score(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return float(1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def scores(y, p):
    y, p = np.asarray(y, np.float64), np.asarray(p, np.float64)
    return {'n': len(y), 'R2': r2_score(y, p) if len(y) > 1 else np.nan,
            'RMSE': float(np.sqrt(((p - y) ** 2).mean())), 'MAE': float(np.abs(p - y).mean())}


def ridge_predictions(X, y, X_eval, alphas):
    """Ridge regression of y on X (intercept not penalised) for every alpha; predictions for X_eval, shape
    (len(alphas), len(X_eval)). w(alpha) = V diag(1 / (s + alpha)) V' Xc' yc with Xc' Xc = V diag(s) V'."""
    X, X_eval, y = np.asarray(X, np.float64), np.asarray(X_eval, np.float64), np.asarray(y, np.float64)
    x_mean, y_mean = X.mean(axis=0), y.mean()
    Xc = X - x_mean
    s, V = np.linalg.eigh(Xc.T @ Xc)
    s = np.clip(s, 0, None)
    Vb = V.T @ (Xc.T @ (y - y_mean))
    P = (X_eval - x_mean) @ V
    return np.stack([y_mean + P @ (Vb / (s + a)) for a in alphas])


def standardize(X_fit, X_other):
    """Both matrices standardised with the mean and standard deviation of X_fit (constant features: scale 1)."""
    X_fit, X_other = np.asarray(X_fit, np.float64), np.asarray(X_other, np.float64)
    if not STANDARDIZE:
        return X_fit, X_other
    mean, std = X_fit.mean(axis=0), X_fit.std(axis=0)
    std[std < 1e-12] = 1.0
    return (X_fit - mean) / std, (X_other - mean) / std


def choose_alpha(X_tr, y_tr, X_va, y_va):
    """The alpha with the lowest validation MSE of a ridge fitted on the training folds, and its validation R2."""
    if ALPHA != 'val':
        return float(ALPHA), np.nan
    Xs_tr, Xs_va = standardize(X_tr, X_va)
    P = ridge_predictions(Xs_tr, y_tr, Xs_va, ALPHAS)
    k = int(np.argmin(((P - y_va) ** 2).mean(axis=1)))
    return float(ALPHAS[k]), r2_score(y_va, P[k])


def fit_predict(X_fit, y_fit, X_eval, alpha):
    """Ridge with this alpha fitted on (X_fit, y_fit), predictions for X_eval."""
    Xs_fit, Xs_eval = standardize(X_fit, X_eval)
    return ridge_predictions(Xs_fit, y_fit, Xs_eval, [alpha])[0]


# check against scikit-learn
try:
    from sklearn.linear_model import Ridge
    rng = np.random.default_rng(0)
    X, Xe = rng.normal(size=(300, 40)) * rng.uniform(0.1, 10, 40), rng.normal(size=(50, 40))
    y = X @ rng.normal(size=40) + rng.normal(size=300) + 30
    ours = ridge_predictions(X, y, Xe, [0.1, 10.0, 1000.0])
    for k, a in enumerate([0.1, 10.0, 1000.0]):
        assert np.allclose(ours[k], Ridge(alpha=a).fit(X, y).predict(Xe), atol=1e-8)
    print('ridge_predictions = scikit-learn Ridge: ok')
except ImportError:
    print('scikit-learn is not installed: check skipped')

# 3. Fit and test

For every row and test fold: the two features files joined on `cluster_id`, then for every feature set the alpha chosen on the validation fold, the ridge fitted on the training and validation folds, and the test predictions. The NL features of a fold are read once for all rows.

In [ ]:
META = ['cluster_id', 'country', 'year', 'rural', 'nl_sensor', 'fold', 'role', TARGET]   # kept with the predictions
NL_DATA = {}


def read_features(path, prefix):
    """features.csv of one model: the other columns, and the features as a float32 matrix."""
    cols = pd.read_csv(path, nrows=0).columns
    fc = [c for c in cols if c.startswith(prefix)]
    assert fc, f'{path} has no feature columns starting with {prefix!r}: check NL_PREFIX and LANDSAT_PREFIX'
    df = pd.read_csv(path, dtype={c: np.float32 for c in fc})
    return df.drop(columns=fc), df[fc].to_numpy()


per_fold, predictions = [], {k: [] for k in ROWS}
for key, f in TODO:
    t0 = time.time()
    if f not in NL_DATA:
        NL_DATA[f] = read_features(nl_file(f), NL_PREFIX)
    nl_meta, X_nl = NL_DATA[f]
    meta, X_ls = read_features(ls_file(key, f), LANDSAT_PREFIX)
    pos = pd.Index(nl_meta['cluster_id']).get_indexer(meta['cluster_id'])    # the 09 clusters in the 08 file
    both = pos >= 0
    meta, X_ls, pos = meta[both].reset_index(drop=True), X_ls[both], pos[both]
    X_nl_rows = X_nl[pos]
    role = meta['role'].fillna('').to_numpy()
    assert (nl_meta['role'].fillna('').to_numpy()[pos] == role).all(), f'{key} test {f}: roles of 08 and 09 differ'
    assert (meta.loc[role == 'test', 'fold'] == f).all()
    y = meta[TARGET].to_numpy(np.float64)
    has_y = np.isfinite(y)
    tr, va, te = has_y & (role == 'train'), has_y & (role == 'val'), has_y & (role == 'test')
    fit = tr | va
    out = meta.loc[te, [c for c in META if c in meta]].reset_index(drop=True)
    row = {'key': key, 'test_fold': f, 'keras_name': KERAS_NAME[f], 'n_train': int(tr.sum()), 'n_val': int(va.sum()),
           'n_test': int(te.sum())}
    for name, X in (('NL + Landsat', np.hstack([X_nl_rows, X_ls])), ('NL', X_nl_rows), ('Landsat', X_ls)):
        alpha, val_r2 = choose_alpha(X[tr], y[tr], X[va], y[va])
        p = fit_predict(X[fit], y[fit], X[te], alpha)
        out[f'pred {name}'] = p
        sc = scores(y[te], p)
        per_fold.append({**row, 'features': name, 'n_features': X.shape[1], 'alpha': alpha,
                         'alpha_at_grid_edge': ALPHA == 'val' and alpha in (ALPHAS[0], ALPHAS[-1]), 'val_r2': val_r2,
                         'test_r2': sc['R2'], 'test_rmse': sc['RMSE'], 'test_mae': sc['MAE']})
    for name, p in (('NL network (08)', nl_meta[f'{TARGET}_pred'].to_numpy()[pos][te]),
                    ('Landsat network (09)', meta[f'{TARGET}_pred'].to_numpy()[te])):
        out[f'pred {name}'] = p
        sc = scores(y[te], p)
        per_fold.append({**row, 'features': name, 'n_features': np.nan, 'alpha': np.nan, 'alpha_at_grid_edge': False,
                         'val_r2': np.nan, 'test_r2': sc['R2'], 'test_rmse': sc['RMSE'], 'test_mae': sc['MAE']})
    predictions[key].append(out)
    r = {x['features']: x['test_r2'] for x in per_fold[-5:]}
    print(f"{key} test {f}: {X_nl_rows.shape[1]} NL + {X_ls.shape[1]} Landsat features, {row['n_train']} + "
          f"{row['n_val']} clusters -> {row['n_test']} | test R2 NL + Landsat "
          f"{r['NL + Landsat']:.3f}, NL {r['NL']:.3f}, Landsat {r['Landsat']:.3f} | networks: NL "
          f"{r['NL network (08)']:.3f}, Landsat {r['Landsat network (09)']:.3f} | alpha "
          f"{per_fold[-5]['alpha']:g} ({time.time() - t0:.0f} s)")
per_fold = pd.DataFrame(per_fold)
edge = per_fold[per_fold['alpha_at_grid_edge']]
if len(edge):
    print(f'NOTE: {len(edge)} alphas at the edge of ALPHAS ({ALPHAS[0]:g} or {ALPHAS[-1]:g}); widen ALPHAS')

# 4. Results

Per row of 09 and feature set: the mean and standard deviation of the test R² over the test folds that are done, and the R², RMSE and MAE (in IWI points) of all their test predictions pooled, also per period (DMSP up to 2013, VIIRS from 2014) and for rural and urban clusters. The first table shows the pooled test R² of every feature set side by side.

In [ ]:
def write_csv(frame, path, **kw):
    tmp = f'{path}.tmp'
    frame.to_csv(tmp, index=False, **kw)
    os.replace(tmp, path)


os.makedirs(OUT_DIR, exist_ok=True)
summary = []
for key in ROWS:
    if not predictions[key]:
        continue
    P = pd.concat(predictions[key], ignore_index=True)
    write_csv(P, os.path.join(OUT_DIR, f'{key}_test_predictions.csv'), float_format='%.6g')
    for name in FEATURE_SETS + NETWORKS:
        pf = per_fold[(per_fold['key'] == key) & (per_fold['features'] == name)]
        rec = {'key': key, 'input': ROWS[key], 'features': name, 'folds done': len(pf),
               'test R2 mean': pf['test_r2'].mean(), 'test R2 sd': pf['test_r2'].std() if len(pf) > 1 else np.nan,
               'median alpha': pf['alpha'].median()}
        sc = scores(P[TARGET], P[f'pred {name}'])
        rec.update({'pooled R2': sc['R2'], 'pooled RMSE': sc['RMSE'], 'pooled MAE': sc['MAE'], 'n test': sc['n']})
        for s, part in P.groupby('nl_sensor'):
            rec[f'pooled R2 {s}'] = scores(part[TARGET], part[f'pred {name}'])['R2']
        if 'rural' in P:
            for label, part in P.groupby(P['rural'].astype(str).str.lower().map({'true': 'rural', 'false': 'urban'})):
                rec[f'pooled R2 {label}'] = scores(part[TARGET], part[f'pred {name}'])['R2']
        summary.append(rec)
summary = pd.DataFrame(summary)
write_csv(per_fold, os.path.join(OUT_DIR, 'ridge_per_fold.csv'))
write_csv(summary, os.path.join(OUT_DIR, 'ridge_results.csv'))

if len(summary):
    print(f'pooled test R2 of the test folds done (ridge alpha: '
          f'{"chosen on the validation fold" if ALPHA == "val" else ALPHA}; standardised features: {STANDARDIZE})')
    side = summary.pivot(index='key', columns='features', values='pooled R2')
    side = side.reindex(index=[k for k in ROWS if k in side.index], columns=FEATURE_SETS + NETWORKS)
    side.insert(0, 'folds', summary[summary['features'] == 'NL + Landsat'].set_index('key')['folds done'])
    display(side.round(3))
else:
    print('no row and test fold with both features files yet')
summary.round(3)

In [ ]:
done = summary[summary['features'] == 'NL + Landsat'].set_index('key') if len(summary) else pd.DataFrame()
if len(done):
    rows = [k for k in ROWS if k in done.index][::-1]
    alone = summary[summary['features'] == 'Landsat'].set_index('key')
    nl_alone = summary.loc[summary['features'] == 'NL', 'test R2 mean'].mean()
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14, 0.5 * len(rows) + 2.2), constrained_layout=True,
                                  gridspec_kw={'width_ratios': [1.5, 1]})
    yy = np.arange(len(rows))
    v, e = done.loc[rows, 'test R2 mean'].to_numpy(), done.loc[rows, 'test R2 sd'].fillna(0).to_numpy()
    handles = [ax.barh(yy, v, xerr=e, height=0.6, color='#2a78d6',
                       error_kw={'ecolor': '#52514e', 'elinewidth': 1, 'capsize': 2},
                       label='ridge on NL + Landsat features: mean ± sd of the test folds')]
    handles.append(ax.scatter(alone.loc[rows, 'test R2 mean'], yy, marker='o', s=40, color='#1baf7a',
                              edgecolors='white', linewidths=1.2, zorder=3, label='ridge on Landsat features alone'))
    handles.append(ax.axvline(nl_alone, color='#52514e', linewidth=1, linestyle='--',
                              label=f'ridge on NL features alone ({nl_alone:.2f})'))
    for k, (vk, ek) in enumerate(zip(v, e)):
        if vk > 0.15:                                      # value inside the bar
            ax.text(0.01, k, f'{vk:.2f}', va='center', ha='left', color='white', fontsize=8)
        else:                                              # small or negative: right of the error bar
            ax.text(min(max(vk + ek, 0), 0.85) + 0.015, k, f'{vk:.2f}', va='center', ha='left', color='#52514e',
                    fontsize=8)
    ax.set_yticks(yy)
    ax.set_yticklabels(rows)
    lo = np.nanmin(np.r_[v, alone.loc[rows, 'test R2 mean'].to_numpy(float), nl_alone])
    ax.set_xlim(max(min(0, lo - 0.05), -1), 1)
    ax.set_xlabel('test R²')
    ax.grid(axis='x', color='#e1e0d9', linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.12 - 0.5 / len(rows)), ncol=1,
              frameon=False, fontsize=8)
    ax.set_title(f'Ridge regression of {TARGET.upper()}, NL (08) and Landsat (09) features', loc='left', fontsize=11)

    full = done[done['folds done'] == done['folds done'].max()]
    best = full['pooled R2'].idxmax()                      # predicted against observed: best row of those with
                                                           # the most test folds done
    P = pd.concat(predictions[best], ignore_index=True)
    obs, pred = P[TARGET].to_numpy(), P['pred NL + Landsat'].to_numpy()
    lo2, hi2 = float(min(obs.min(), pred.min())), float(max(obs.max(), pred.max()))
    lo2, hi2 = lo2 - 0.03 * (hi2 - lo2), hi2 + 0.03 * (hi2 - lo2)
    blues = LinearSegmentedColormap.from_list('blues', ['#cde2fb', '#86b6ef', '#3987e5', '#256abf', '#184f95',
                                                        '#0d366b'])          # one hue, light (few) to dark (many)
    hb = ax2.hexbin(obs, pred, gridsize=40, extent=(lo2, hi2, lo2, hi2), mincnt=1, cmap=blues, linewidths=0.2,
                    edgecolors='face')
    fig.colorbar(hb, ax=ax2, shrink=0.85, label='clusters', ticks=MaxNLocator(integer=True))
    ax2.plot([lo2, hi2], [lo2, hi2], color='#52514e', linewidth=1, linestyle='--')
    ax2.text(0.98, 0.02, '1:1', transform=ax2.transAxes, ha='right', va='bottom', fontsize=8, color='#52514e')
    ax2.set_xlim(lo2, hi2)
    ax2.set_ylim(lo2, hi2)
    ax2.set_aspect('equal')
    ax2.set_xlabel(f'observed {TARGET.upper()}')
    ax2.set_ylabel(f'predicted {TARGET.upper()}')
    n_folds = int(done.at[best, 'folds done'])
    ax2.text(0.03, 0.97, f"pooled R² {done.at[best, 'pooled R2']:.2f}, RMSE {done.at[best, 'pooled RMSE']:.1f}\n"
             f"{n_folds} test fold{'s' * (n_folds > 1)}, {len(P)} clusters", transform=ax2.transAxes, va='top',
             fontsize=8, color='#0b0b0b')
    ax2.set_title(f'NL + {best}: test predictions pooled', loc='left', fontsize=11)
    for a in (ax, ax2):
        for side in ('top', 'right'):
            a.spines[side].set_visible(False)
    png = os.path.join(OUT_DIR, 'ridge_r2.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no results yet')

# 5. Results per country

The pooled test predictions of section 4 by country: every cluster once, predicted by the models for which its fold was the test fold (03 spreads the clusters of every country over the five folds, so each country is predicted by all five models). For every row, feature set and country, `ridge_per_country.csv` holds:

* `n test`, and the mean and standard deviation of the observed IWI;
* `R2` against the country's own mean IWI (negative when the predictions miss the country's level or spread), and `r2`, the squared correlation of predicted and observed IWI within the country (the r² of Yeh et al. 2020, which an offset does not lower);
* `RMSE`, `MAE` and `bias` (the mean of predicted − observed), in IWI points;
* `fold R2 mean` and `fold R2 sd`: the R² within the country in each test fold that holds at least `MIN_FOLD_CLUSTERS` of its clusters (`folds`: how many such folds).

The tables and the plot show the countries with at least `MIN_COUNTRY_CLUSTERS` test clusters for the row `COUNTRY_ROW` (`'best'`: the row of the plot in section 4): the scores of the ridge on NL + Landsat features, the R² of every feature set, and the R² of NL + Landsat for every row; their last two lines are the mean and the median over these countries. The plot shows the R² per country and the bias of the ridge on NL + Landsat features (`ridge_per_country.png`).

In [ ]:
def country_scores(part, name):
    """Test scores of one feature set for the clusters of one country (part: its rows of the pooled predictions)."""
    y, p = part[TARGET].to_numpy(np.float64), part[f'pred {name}'].to_numpy(np.float64)
    spread = len(y) > 1 and y.std() > 0
    by_fold = [r2_score(fp[TARGET], fp[f'pred {name}']) for _, fp in part.groupby('fold')
               if len(fp) >= MIN_FOLD_CLUSTERS and fp[TARGET].std() > 0]
    return {'n test': len(y), f'{TARGET} mean': y.mean(), f'{TARGET} sd': y.std(ddof=1) if len(y) > 1 else np.nan,
            'R2': r2_score(y, p) if spread else np.nan,
            'r2': float(np.corrcoef(y, p)[0, 1] ** 2) if spread and len(y) > 2 and p.std() > 0 else np.nan,
            'RMSE': float(np.sqrt(((p - y) ** 2).mean())), 'MAE': float(np.abs(p - y).mean()),
            'bias': float((p - y).mean()), 'folds': len(by_fold),
            'fold R2 mean': float(np.mean(by_fold)) if by_fold else np.nan,
            'fold R2 sd': float(np.std(by_fold, ddof=1)) if len(by_fold) > 1 else np.nan}


def with_summary(table):
    """The table with the mean and the median over its countries as two more lines."""
    stats = table.drop(columns=['n test'], errors='ignore').agg(['mean', 'median'])
    stats.index = ['mean of these countries', 'median of these countries']
    out = pd.concat([table, stats])
    if 'n test' in out:
        out['n test'] = out['n test'].astype('Int64')
    return out


per_country = []
for key in ROWS:
    if predictions[key]:
        P = pd.concat(predictions[key], ignore_index=True)
        for name in FEATURE_SETS + NETWORKS:
            for country, part in P.groupby('country'):
                per_country.append({'key': key, 'features': name, 'country': country,
                                    'test folds done': P['fold'].nunique(), **country_scores(part, name)})
per_country = pd.DataFrame(per_country)
write_csv(per_country, os.path.join(OUT_DIR, 'ridge_per_country.csv'))

country_row = None
if len(per_country):
    nl_ls = summary[summary['features'] == 'NL + Landsat'].set_index('key')
    if COUNTRY_ROW == 'best':                              # as in the plot of section 4
        full = nl_ls[nl_ls['folds done'] == nl_ls['folds done'].max()]
        country_row = full['pooled R2'].idxmax()
    else:
        assert COUNTRY_ROW in nl_ls.index, f'COUNTRY_ROW = {COUNTRY_ROW!r} is not a row with results'
        country_row = COUNTRY_ROW
    big = per_country[per_country['n test'] >= MIN_COUNTRY_CLUSTERS]
    mine = big[big['key'] == country_row]
    detail = (mine[mine['features'] == 'NL + Landsat'].set_index('country')
              .drop(columns=['key', 'features', 'test folds done']).sort_values('R2', ascending=False))
    by_set = mine.pivot(index='country', columns='features', values='R2').reindex(index=detail.index,
                                                                                  columns=FEATURE_SETS + NETWORKS)
    by_set.insert(0, 'n test', detail['n test'])
    by_row = big[big['features'] == 'NL + Landsat'].pivot(index='country', columns='key', values='R2')
    by_row = by_row.reindex(index=list(detail.index) + sorted(set(by_row.index) - set(detail.index)),
                            columns=[k for k in ROWS if k in by_row.columns])
    small = per_country[(per_country['key'] == country_row) & (per_country['features'] == 'NL + Landsat') &
                        (per_country['n test'] < MIN_COUNTRY_CLUSTERS)]
    print(f'{country_row}: test results per country of the ridge on NL + Landsat features, {len(detail)} '
          f'countries with at least {MIN_COUNTRY_CLUSTERS} test clusters' +
          (f' (left out: {", ".join(f"{c} {n}" for c, n in zip(small["country"], small["n test"]))})' if len(small)
           else ''))
    display(detail.round({f'{TARGET} mean': 1, f'{TARGET} sd': 1, 'R2': 3, 'r2': 3, 'RMSE': 1, 'MAE': 1, 'bias': 1,
                          'fold R2 mean': 3, 'fold R2 sd': 3}))
    print(f'{country_row}: test R2 per country and feature set')
    display(with_summary(by_set).round(3))
    print('ridge on NL + Landsat features: test R2 per country and row')
    display(with_summary(by_row).round(3))
else:
    print('no test predictions yet')

In [ ]:
if country_row is not None and len(detail):
    show = by_set.iloc[::-1]                               # best country on top
    yy = np.arange(len(show))
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 0.32 * len(show) + 2.3), sharey=True, constrained_layout=True,
                                  gridspec_kw={'width_ratios': [1.7, 1]})
    v = show['NL + Landsat'].to_numpy(float)
    handles = [ax.barh(yy, v, height=0.6, color='#2a78d6', label='ridge on NL + Landsat features')]
    handles.append(ax.scatter(show['Landsat'], yy, marker='o', s=34, color='#1baf7a', edgecolors='white',
                              linewidths=1.0, zorder=3, label='ridge on Landsat features alone'))
    handles.append(ax.scatter(show['NL'], yy, marker='|', s=110, color='#52514e', linewidths=1.6, zorder=3,
                              label='ridge on NL features alone'))
    marks = [v, show['Landsat'].to_numpy(float), show['NL'].to_numpy(float)]
    x0 = max(min(0.0, np.nanmin(np.r_[np.concatenate(marks), 0.0]) - 0.05), -1.0)   # R2 below -1: cut there
    for k, vk in enumerate(v):
        if vk < x0:                                        # the value of a cut bar
            ax.text(x0 + 0.01, k, f'{vk:.2f}', va='center', ha='left', color='white', fontsize=7)
    ax.set_xlim(x0, 1)
    ax.axvline(0, color='#52514e', linewidth=0.8)
    ax.set_yticks(yy)
    ax.set_yticklabels([f'{c} ({n})' for c, n in zip(show.index, show['n test'])], fontsize=8)
    ax.set_xlabel('test R² within the country')
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.55 / (0.32 * len(show) + 1.1)),
              ncol=3, frameon=False, fontsize=8)
    ax.set_title(f'{country_row}: test R² per country (test clusters in brackets)', loc='left', fontsize=11)
    ax2.barh(yy, detail.loc[show.index, 'bias'].to_numpy(float), height=0.6, color='#86b6ef')
    ax2.axvline(0, color='#52514e', linewidth=0.8)
    ax2.set_xlabel(f'predicted − observed {TARGET.upper()}, ridge on NL + Landsat features')
    ax2.set_title('mean error per country', loc='left', fontsize=11)
    for a in (ax, ax2):
        a.grid(axis='x', color='#e1e0d9', linewidth=0.8)
        a.set_axisbelow(True)
        for spine in ('top', 'right'):
            a.spines[spine].set_visible(False)
    png = os.path.join(OUT_DIR, 'ridge_per_country.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no results per country yet')

# 6. NDVI: whether and where it can help

A cheap check with the test predictions of section 4, before more modelling of NDVI.

* **NDVI statistics of every cluster**, from the band caches of 04 to 10 (`NDVI_CACHES`; read once, then kept in `ridge_runs/ndvi_patch_stats_<cache>.csv` for 11 and 12): the patch mean and standard deviation of the 12 monthly NDVI bands and of the 6 wavelet bands, and the mean, minimum, maximum, range, standard deviation and annual and semi-annual harmonics (cos, sin) of the patch-mean NDVI series, 45 numbers.
* **Residual screen:** the test residuals of the ridge on NL + Landsat features (observed − predicted IWI, every cluster from the model of its test fold) are regressed on these statistics with the ridge of section 2 and the same folds (alpha on the validation fold, fitted on the training and validation folds, predicted for the test fold). The NDVI terms of this ridge (without its intercept, the mean residual, so that shifting all predictions does not count) added to the prediction give `NL + Landsat + NDVI`. If NDVI holds wealth information that the NL and Landsat features miss, its test R² is higher. `NDVI x rural` fits separate slopes for rural and urban clusters (the statistics centred within each, so that the rural-urban difference of the residuals is not a term): NDVI may say more about rural than about urban wealth. The residuals of the training clusters come from models that saw the test fold, as in stacking: this is a screen, not a final model.
* **Rows with NDVI in the network:** the ridge on NL + Landsat features of every other row (NDVI series, wavelet bands) against the `landsat` row, on the same test clusters.
* **Where:** the change of the test R² (`dR2`) for all clusters, rural and urban clusters, dark clusters (the mean of the NL image at most `DARK_NL` for its sensor, where NL tells nothing) and lit ones, within countries (the R² of the deviations from the country means, so that a country's level does not count) and for rural clusters within countries; and per country. The 95 % interval comes from `BOOTSTRAP` resamples of 03's spatial groups (clusters closer than 9.5 km are drawn together), and `folds dR2 > 0` counts the test folds with a gain.
* **Correlations** of the residuals with every NDVI statistic, over all clusters, within countries and for rural clusters within countries, for `SCREEN_ROW`.

Rows need all five test folds. The results go to `ndvi_screen.csv` (every comparison and subset), `ndvi_screen_per_country.csv`, `ndvi_screen_correlations.csv`, `ndvi_screen_predictions.csv` (every test cluster of every row: the prediction, the two NDVI corrections, rural, dark) and `ndvi_screen.png`.

In [ ]:
import multiprocessing

NDVI_CHANNELS = [f'NDVI_{k:02d}' for k in range(1, 13)] + ['LFC_min', 'LFC_max', 'LFC_mean', 'HFC_min', 'HFC_max',
                                                           'HFC_mean']
FIRST_NDVI = 7                                    # band cache channels: 7 Landsat, 12 monthly NDVI, 6 wavelet bands


def _patch_moments(task):
    """Patch mean and standard deviation of the NDVI and wavelet bands of the cache rows a to b."""
    path, a, b = task
    x = np.load(path, mmap_mode='r')[a:b, FIRST_NDVI:FIRST_NDVI + len(NDVI_CHANNELS)].astype(np.float32)
    return a, x.mean(axis=(2, 3), dtype=np.float64), x.std(axis=(2, 3), dtype=np.float64)


def ndvi_patch_stats(cache_dir):
    """The NL mean and the NDVI statistics of every cluster of a band cache (computed once, then read from the CSV)."""
    out = os.path.join(WORK_DIR, 'ridge_runs', f'ndvi_patch_stats_{os.path.basename(cache_dir)}.csv')
    index = pd.read_csv(os.path.join(cache_dir, 'index.csv'))
    if os.path.exists(out):
        stats = pd.read_csv(out)
        if stats['cluster_id'].tolist() == index['cluster_id'].tolist():
            print('NDVI statistics found:', out)
            return stats
    path = os.path.join(cache_dir, 'features.npy')
    shape = np.load(path, mmap_mode='r').shape
    assert shape[:2] == (len(index), FIRST_NDVI + len(NDVI_CHANNELS)), (cache_dir, shape)
    t0 = time.time()
    mean, sd = (np.full((shape[0], len(NDVI_CHANNELS)), np.nan) for _ in range(2))
    with multiprocessing.get_context('fork').Pool(N_WORKERS) as pool:
        for a, m, s in pool.imap_unordered(_patch_moments, [(path, a, min(a + 32, shape[0]))
                                                            for a in range(0, shape[0], 32)]):
            mean[a:a + len(m)], sd[a:a + len(m)] = m, s
    stats = index[['cluster_id', 'nl_mean', 'ok']].copy()
    for j, c in enumerate(NDVI_CHANNELS):
        stats[f'{c} mean'], stats[f'{c} sd'] = mean[:, j], sd[:, j]
    curve, t = mean[:, :12], 2 * np.pi * np.arange(12) / 12      # the patch-mean NDVI of the 12 months
    stats['NDVI year mean'], stats['NDVI year min'], stats['NDVI year max'] = curve.mean(1), curve.min(1), curve.max(1)
    stats['NDVI year range'], stats['NDVI year sd'] = curve.max(1) - curve.min(1), curve.std(1)
    for k in (1, 2):                                        # annual and semi-annual harmonics (time 0: January)
        stats[f'NDVI cos{k}'], stats[f'NDVI sin{k}'] = curve @ np.cos(k * t) / 6, curve @ np.sin(k * t) / 6
    stats.loc[~stats['ok'].astype(bool), stats.columns[3:]] = np.nan
    os.makedirs(os.path.dirname(out), exist_ok=True)
    write_csv(stats, out, float_format='%.6g')
    print(f'NDVI statistics of {shape[0]} clusters in {time.time() - t0:.0f} s -> {out}')
    return stats


parts = []
for s, d in NDVI_CACHES.items():
    if os.path.exists(os.path.join(d, 'index.csv')) and os.path.exists(os.path.join(d, 'features.npy')):
        parts.append(ndvi_patch_stats(d).assign(nl_sensor=s))
    else:
        print(f'no band cache {d}: its clusters are left out of section 6')
NDVI = (pd.concat(parts, ignore_index=True) if parts else
        pd.DataFrame(columns=['cluster_id', 'nl_mean', 'ok', 'nl_sensor']))
assert NDVI['cluster_id'].is_unique, 'a cluster is in two band caches'
NDVI_BLOCK = [c for c in NDVI.columns if c.startswith(('NDVI', 'LFC', 'HFC'))]
NDVI['dark'] = NDVI['nl_mean'] <= NDVI['nl_sensor'].map(DARK_NL)
groups = pd.read_csv(FOLDS_CSV, usecols=lambda c: c in ('cluster_id', 'group')) if os.path.exists(FOLDS_CSV) else None
if groups is not None and 'group' in groups:              # clusters closer than 9.5 km: one group (and one fold)
    UNIT, unit_name = groups.set_index('cluster_id')['group'], "03's spatial groups"
else:
    UNIT, unit_name = pd.Series(dtype=float), 'clusters (no groups in FOLDS_CSV)'
ok = NDVI['ok'].astype(bool)
print(f'{len(NDVI_BLOCK)} NDVI statistics of {ok.sum()} clusters; dark (NL mean at most {DARK_NL}):',
      {s: f"{g.loc[g['ok'].astype(bool), 'dark'].mean():.0%} of {g['ok'].astype(bool).sum()}"
       for s, g in NDVI.groupby('nl_sensor')}, f'| bootstrap: {BOOTSTRAP} resamples of {unit_name}')

In [ ]:
def delta_r2(y, p_old, p_new, unit, country, counts):
    """R2 of p_old and of p_new and their difference, for every row of counts (how often each unit is drawn; a row of
    ones: the data). With country: the R2 of the deviations from the country means (within countries)."""
    d_old, d_new = y - p_old, y - p_new
    lab = np.zeros(len(y), int) if country is None else pd.factorize(country)[0]
    sst, sse_old, sse_new = (np.zeros(len(counts)) for _ in range(3))
    for g in range(lab.max() + 1 if len(lab) else 0):
        m = lab == g
        u, inv = np.unique(unit[m], return_inverse=True)
        S = np.zeros((len(u), 7))
        np.add.at(S, inv, np.stack([np.ones(m.sum()), y[m], y[m] ** 2, d_old[m], d_old[m] ** 2, d_new[m],
                                    d_new[m] ** 2], axis=1))
        for a in range(0, len(counts), 250):               # blocks of resamples
            T = counts[a:a + 250][:, u] @ S
            n = np.maximum(T[:, 0], 1e-12)
            sst[a:a + 250] += T[:, 2] - T[:, 1] ** 2 / n
            if country is None:
                sse_old[a:a + 250] += T[:, 4]
                sse_new[a:a + 250] += T[:, 6]
            else:                                          # the residuals' country means removed
                sse_old[a:a + 250] += T[:, 4] - T[:, 3] ** 2 / n
                sse_new[a:a + 250] += T[:, 6] - T[:, 5] ** 2 / n
    with np.errstate(invalid='ignore', divide='ignore'):
        r_old, r_new = 1 - sse_old / sst, 1 - sse_new / sst
    return r_new - r_old, r_old, r_new


def rural_urban(P):
    """Rural and urban clusters (a cluster without the information is neither)."""
    r = P['rural'].astype(str).str.lower()
    return r.eq('true').to_numpy(), r.eq('false').to_numpy()


def subsets(P):
    """name -> (clusters, within countries)."""
    (rural, urban), dark, every = rural_urban(P), P['dark'].to_numpy(bool), np.ones(len(P), bool)
    return {'all': (every, False), 'rural': (rural, False), 'urban': (urban, False), 'dark': (dark, False),
            'lit': (~dark, False), 'within country': (every, True), 'rural, within country': (rural, True)}


def paired(P, old, new, comparison, key):
    """dR2 = R2(new) - R2(old) on the same clusters in every subset, with the bootstrap interval and the test folds."""
    y, po, pn = (P[c].to_numpy(np.float64) for c in (TARGET, old, new))
    unit, country, fold = P['unit'].to_numpy(), P['country'].to_numpy(), P['fold'].to_numpy()
    out = []
    for name, (m, within) in subsets(P).items():
        if m.sum() < 10:
            continue
        part = lambda k: (y[k], po[k], pn[k], unit[k], country[k] if within else None)
        d, r_old, r_new = (v[0] for v in delta_r2(*part(m), ONES))
        rec = {'comparison': comparison, 'key': key, 'subset': name, 'n': int(m.sum()), 'R2 before': r_old,
               'R2 after': r_new, 'dR2': d}
        if BOOTSTRAP:
            b = delta_r2(*part(m), COUNTS)[0]
            rec.update({'dR2 low': np.nanquantile(b, 0.025), 'dR2 high': np.nanquantile(b, 0.975),
                        'P(dR2 > 0)': float(np.mean(b > 0))})
        gains = [delta_r2(*part(m & (fold == f)), ONES)[0][0] for f in FOLDS if (m & (fold == f)).sum() >= 10]
        rec['folds dR2 > 0'] = f'{sum(g > 0 for g in gains)} of {len(gains)}'
        out.append(rec)
    return out


def design(screen_name, X, rural, urban, fit):
    """The features of a screen: the NDVI statistics ('NDVI'), or the statistics of the rural clusters and those of the
    urban clusters as separate columns, each centred on its clusters in fit ('NDVI x rural'): separate slopes, and the
    rural-urban difference of the residuals is not a term."""
    if screen_name == 'NDVI':
        return X
    part = lambda m: (X - (X[fit & m].mean(axis=0) if (fit & m).any() else 0)) * m[:, None]
    return np.hstack([part(rural), part(urban)])


# the screens: ridge of the test residuals of NL + Landsat on the NDVI statistics (folds of the networks)
SCREENS = {'NDVI': 'pred NL + Landsat + NDVI', 'NDVI x rural': 'pred NL + Landsat + NDVI x rural'}
complete_rows = [k for k in ROWS if predictions[k] and len(per_fold) and per_fold.loc[
    (per_fold['key'] == k) & (per_fold['features'] == 'NL + Landsat'), 'test_fold'].nunique() == len(FOLDS)]
SCREEN = {}
for key in complete_rows if len(NDVI_BLOCK) else []:
    P = pd.concat(predictions[key], ignore_index=True).merge(NDVI[['cluster_id', 'dark'] + NDVI_BLOCK], on='cluster_id')
    P = P[P[NDVI_BLOCK].notna().all(axis=1)].reset_index(drop=True)
    e, X = (P[TARGET] - P['pred NL + Landsat']).to_numpy(np.float64), P[NDVI_BLOCK].to_numpy(np.float64)
    rural, urban = rural_urban(P)
    correction, alphas = {s: np.zeros(len(P)) for s in SCREENS}, {s: [] for s in SCREENS}
    for k, f in enumerate(FOLDS):
        te, va = (P['fold'] == f).to_numpy(), (P['fold'] == FOLDS[(k - 1) % len(FOLDS)]).to_numpy()
        tr, fit = ~te & ~va, ~te
        for s in SCREENS:
            Z_tr, Z_fit = design(s, X, rural, urban, tr), design(s, X, rural, urban, fit)
            alpha, _ = choose_alpha(Z_tr[tr], e[tr], Z_tr[va], e[va])
            # the NDVI terms only: without the intercept (the mean residual), a shift of all predictions is not NDVI
            correction[s][te] = fit_predict(Z_fit[fit], e[fit], Z_fit[te], alpha) - e[fit].mean()
            alphas[s].append(alpha)
    for s, col in SCREENS.items():
        P[f'{s} correction'], P[col] = correction[s], P['pred NL + Landsat'] + correction[s]
    SCREEN[key] = P
    print(f'{key}: {len(P)} test clusters; share of the squared test error removed:',
          ', '.join(f'{s} {1 - ((e - c) ** 2).sum() / (e ** 2).sum():.2%}' for s, c in correction.items()),
          '| alphas:', {s: [f'{a:g}' for a in v] for s, v in alphas.items()})

screen, screen_country, screen_corr = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
if SCREEN:
    ids = pd.concat([P['cluster_id'] for P in SCREEN.values()]).unique()
    u = UNIT.reindex(ids).to_numpy(object)
    u[pd.isna(u)] = [f'cluster {c}' for c in ids[pd.isna(u)]]  # without a group: a unit of its own
    unit_of = pd.Series(pd.factorize(u)[0], index=ids)
    for P in SCREEN.values():
        P['unit'] = unit_of.reindex(P['cluster_id']).to_numpy()
    ONES = np.ones((1, unit_of.max() + 1))
    rng = np.random.default_rng(0)
    COUNTS = np.vstack([rng.multinomial(len(ONES[0]), ONES[0] / len(ONES[0]), size=min(250, BOOTSTRAP - a))
                        .astype(np.int32) for a in range(0, BOOTSTRAP, 250)]) if BOOTSTRAP else None
    out = []
    for key, P in SCREEN.items():
        for s, col in SCREENS.items():
            out += paired(P, 'pred NL + Landsat', col, f'{s} screen', key)
    if 'landsat' in SCREEN:                                 # the rows with NDVI in the network, against landsat
        base = SCREEN['landsat'][['cluster_id', 'pred NL + Landsat']].rename(columns={'pred NL + Landsat': 'landsat'})
        for key, P in SCREEN.items():
            if key != 'landsat':
                out += paired(P.merge(base, on='cluster_id'), 'landsat', 'pred NL + Landsat', 'row vs landsat', key)
    screen = pd.DataFrame(out)
    rec = []                                     # per country: R2 against the country mean (section 5) and within
    for key, P in SCREEN.items():
        for country, g in P.groupby('country'):
            y = g[TARGET].to_numpy(np.float64)
            d_old, sst = y - g['pred NL + Landsat'].to_numpy(), ((y - y.mean()) ** 2).sum()
            if len(y) < 3 or sst == 0:
                continue
            r = {'key': key, 'country': country, 'n test': len(y), 'rural share': rural_urban(g)[0].mean(),
                 'dark share': g['dark'].mean(), 'R2 before': 1 - (d_old ** 2).sum() / sst}
            for s, col in SCREENS.items():
                d_new = y - g[col].to_numpy()
                r[f'dR2 ({s})'] = ((d_old ** 2).sum() - (d_new ** 2).sum()) / sst
                r[f'dR2 within ({s})'] = (((d_old - d_old.mean()) ** 2).sum()
                                          - ((d_new - d_new.mean()) ** 2).sum()) / sst
            rec.append({**r, **{f'mean {s} correction': g[f'{s} correction'].mean() for s in SCREENS}})
    screen_country = pd.DataFrame(rec)
    row = SCREEN_ROW if SCREEN_ROW in SCREEN else next(iter(SCREEN))
    P = SCREEN[row].copy()                                  # correlations of the residuals with the NDVI statistics
    P['residual'] = P[TARGET] - P['pred NL + Landsat']
    cols = ['residual'] + NDVI_BLOCK
    W = P[cols] - P.groupby('country')[cols].transform('mean')
    rural = rural_urban(P)[0]
    R = P.loc[rural, cols] - P[rural].groupby('country')[cols].transform('mean')
    screen_corr = pd.DataFrame({'statistic': NDVI_BLOCK, 'r': [P['residual'].corr(P[c]) for c in NDVI_BLOCK],
                                'r within country': [W['residual'].corr(W[c]) for c in NDVI_BLOCK],
                                'r rural, within country': [R['residual'].corr(R[c]) for c in NDVI_BLOCK]})
    screen_corr = screen_corr.reindex(screen_corr['r within country'].abs().sort_values(ascending=False).index)
    write_csv(screen, os.path.join(OUT_DIR, 'ndvi_screen.csv'))
    write_csv(screen_country, os.path.join(OUT_DIR, 'ndvi_screen_per_country.csv'))
    write_csv(screen_corr.assign(key=row), os.path.join(OUT_DIR, 'ndvi_screen_correlations.csv'))
    keep = ['cluster_id', 'country', 'year', 'nl_sensor', 'fold', 'rural', 'dark', TARGET, 'pred NL + Landsat',
            'NDVI correction', 'NDVI x rural correction']
    write_csv(pd.concat([Q[[c for c in keep if c in Q]].assign(key=k) for k, Q in SCREEN.items()]),
              os.path.join(OUT_DIR, 'ndvi_screen_predictions.csv'), float_format='%.6g')

    fmt = lambda r: (f"{r['dR2']:+.3f}" + (f" [{r['dR2 low']:+.3f}, {r['dR2 high']:+.3f}]" if BOOTSTRAP else ''))
    names = list(subsets(SCREEN[row]))
    side = screen.assign(cell=screen.apply(fmt, axis=1), what=screen['comparison'] + ': ' + screen['key'])
    side = side.pivot(index='what', columns='subset', values='cell')
    order = [f'{s} screen: {k}' for k in SCREEN for s in SCREENS]
    order += [f'row vs landsat: {k}' for k in SCREEN if k != 'landsat' and 'landsat' in SCREEN]
    print('change of the test R2 of NL + Landsat, dR2 [95 % interval]; NDVI screen: with the NDVI correction; '
          'NDVI x rural: separate slopes for rural and urban clusters; row vs landsat: that row against landsat')
    display(side.reindex(index=order, columns=names))
    for s in SCREENS:
        print(f'{row}, {s} screen:')
        display(screen[(screen['comparison'] == f'{s} screen') & (screen['key'] == row)]
                .drop(columns=['comparison', 'key']).set_index('subset').round(4))
    pc = screen_country[(screen_country['key'] == row) & (screen_country['n test'] >= MIN_COUNTRY_CLUSTERS)]
    print(f'{row}, the screens per country ({len(pc)} countries with at least {MIN_COUNTRY_CLUSTERS} test clusters; '
          'dR2 against the country mean as in section 5, dR2 within: the country level removed):')
    display(pc.drop(columns='key').set_index('country').sort_values('dR2 within (NDVI)', ascending=False).round(3))
    print(f'{row}: correlation of the residuals of NL + Landsat with the NDVI statistics '
          '(strongest within countries first)')
    display(screen_corr.head(12).set_index('statistic').round(3))
else:
    print('no row with all five test folds and NDVI statistics yet')

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

if len(screen):
    show = [('NDVI screen', row, '#2a78d6', 'o', f'NDVI correction ({row})'),
            ('NDVI x rural screen', row, '#184f95', 's', f'NDVI correction, rural and urban slopes ({row})')]
    if 'landsat_ndvi' in SCREEN and 'landsat' in SCREEN:
        show.append(('row vs landsat', 'landsat_ndvi', '#1baf7a', 'D', 'NDVI series in the network: landsat_ndvi '
                     'against landsat'))
    pc = screen_country[(screen_country['key'] == row) & (screen_country['n test'] >= MIN_COUNTRY_CLUSTERS)]
    pc = pc.sort_values('dR2 within (NDVI)')
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, max(4.8, 0.3 * len(pc) + 2.3)), constrained_layout=True,
                                  gridspec_kw={'width_ratios': [1, 1.15]})
    yy = np.arange(len(names))[::-1]
    for j, (comp, key, color, marker, label) in enumerate(show):
        t = screen[(screen['comparison'] == comp) & (screen['key'] == key)].set_index('subset').reindex(names)
        off = 0.18 * (j - (len(show) - 1) / 2)
        err = [t['dR2'] - t['dR2 low'], t['dR2 high'] - t['dR2']] if BOOTSTRAP else None
        ax.errorbar(t['dR2'], yy - off, xerr=err, fmt=marker, color=color, ecolor=color, elinewidth=1.2, capsize=2,
                    markersize=5.5, label=label)
    ax.axvline(0, color='#52514e', linewidth=0.8)
    ax.set_yticks(yy)
    ax.set_yticklabels(names)
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel('change of the test R² of NL + Landsat' + (' (95 % interval)' if BOOTSTRAP else ''))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=8)
    ax.set_title('Does NDVI add to NL + Landsat?', loc='left', fontsize=11)
    if len(pc):
        y2 = np.arange(len(pc))
        v = pc['dR2 within (NDVI)'].to_numpy(float)
        ax2.barh(y2, v, height=0.6, color=np.where(v >= 0, '#2a78d6', '#eb6834'))
        ax2.scatter(pc['dR2 within (NDVI x rural)'], y2, marker='s', s=26, color='#184f95', edgecolors='white',
                    linewidths=0.8, zorder=3)
        ax2.set_yticks(y2)
        ax2.set_yticklabels([f'{c} ({n})' for c, n in zip(pc['country'], pc['n test'])], fontsize=8)
        ax2.axvline(0, color='#52514e', linewidth=0.8)
        ax2.set_xlabel('change of the test R² within the country (its level removed)')
        ax2.legend(handles=[Patch(color='#2a78d6', label='NDVI correction: gain'),
                            Patch(color='#eb6834', label='NDVI correction: loss'),
                            Line2D([], [], marker='s', linestyle='', color='#184f95', markeredgecolor='white',
                                   label='rural and urban slopes')],
                   loc='upper center', bbox_to_anchor=(0.5, -0.12), frameon=False, fontsize=8, ncol=3)
    ax2.set_title(f'The NDVI screen per country ({row}, test clusters in brackets)', loc='left', fontsize=11)
    for a in (ax, ax2):
        a.grid(axis='x', color='#e1e0d9', linewidth=0.8)
        a.set_axisbelow(True)
        for spine in ('top', 'right'):
            a.spines[spine].set_visible(False)
    png = os.path.join(OUT_DIR, 'ndvi_screen.png')
    fig.savefig(png + '.tmp.png', dpi=150)
    os.replace(png + '.tmp.png', png)
    plt.show()
else:
    print('no NDVI screen yet')